# Datos totales. Viendo qué filtramos

In [ ]:
from datasets import load_dataset
import pandas as pd

# 1. Cargar el dataset CLINC150 (usamos la versión 'plus' que tiene todos los datos)
print("Cargando el dataset...")
dataset = load_dataset("clinc_oos", 'plus')
dataset

Cargando el dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['text', 'intent'],
        num_rows: 15250
    })
    validation: Dataset({
        features: ['text', 'intent'],
        num_rows: 3100
    })
    test: Dataset({
        features: ['text', 'intent'],
        num_rows: 5500
    })
})

Vale el dataset viene partido en 3, pero tenemos pocos datos. Opciones:
1. Juntamos los tres datos y la forma de validar sería haciendo validación cruzada (obtenemos por tanto una proyección del rendimiento del modelo y no una evaluación en sí misma), pero tenemos más datos para entrenar en total
2. Juntamos dos de tres conjuntos de datos (diría que los que tengan más datos, o sea train + test/val el más abundante), hacemos validación cruzada para optimización de hiperparámetros sobre esos datos combinados, y sobre el restante es nuestra métrica de evaluación real. Pero tendríamos menos datos para entrenar.

Optamos por 2.

In [ ]:
df_train = dataset['train'].to_pandas()
df_val = dataset['validation'].to_pandas()
df_test = dataset['test'].to_pandas()

Veamos las intenciones que hay.

In [ ]:
nombres_intenciones = dataset['train'].features['intent'].names
nombres_intenciones[:20]

['restaurant_reviews',
 'nutrition_info',
 'account_blocked',
 'oil_change_how',
 'time',
 'weather',
 'redeem_rewards',
 'interest_rate',
 'gas_type',
 'accept_reservations',
 'smart_home',
 'user_name',
 'report_lost_card',
 'repeat',
 'whisper_mode',
 'what_are_your_hobbies',
 'order',
 'jump_start',
 'schedule_meeting',
 'meeting_schedule']

In [ ]:
# Filtramos las de cocina (Gemini)
intenciones_cocina = [
    'meal_suggestion',         # Sugerencia de comida
    'recipe',                  # Búsqueda de recetas
    'ingredients_list',        # Ingredientes de una receta
    'ingredient_substitution', # Sustitución de ingredientes
    'nutrition_info',          # Información nutricional
    'calories',                # Calorías
    'cook_time',               # Tiempo de cocción
    'food_last',               # Caducidad
]

Filtramos datasets.

In [ ]:
# Creamos una nueva columna con el nombre de la intención en texto
df_train['intent_name'] = df_train['intent'].apply(lambda x: nombres_intenciones[x])
df_val['intent_name'] = df_val['intent'].apply(lambda x: nombres_intenciones[x])
df_test['intent_name'] = df_test['intent'].apply(lambda x: nombres_intenciones[x])

# Filtramos. Nos quedamos solo con las filas cuya intención esté en nuestra lista
cocina_train = df_train[df_train['intent_name'].isin(intenciones_cocina)]
cocina_val = df_val[df_val['intent_name'].isin(intenciones_cocina)]
cocina_test = df_test[df_test['intent_name'].isin(intenciones_cocina)]

# Resultados
print("Train")
print(f"Total de ejemplos originales: {len(df_train)}")
print(f"Total de ejemplos de cocina: {len(cocina_train)}")

print("\nVal")
print(f"Total de ejemplos originales: {len(df_val)}")
print(f"Total de ejemplos de cocina: {len(cocina_val)}")

print("\nTest")
print(f"Total de ejemplos originales: {len(df_test)}")
print(f"Total de ejemplos de cocina: {len(cocina_test)}")

Train
Total de ejemplos originales: 15250
Total de ejemplos de cocina: 800

Val
Total de ejemplos originales: 3100
Total de ejemplos de cocina: 160

Test
Total de ejemplos originales: 5500
Total de ejemplos de cocina: 240


## Dataset de cocina. Balanceo de clases

In [ ]:
cocina_train

,text,intent,intent_name
5500,what's the nutritional info for spaghetti,1,nutrition_info
5501,what's the nutritional info for pizza,1,nutrition_info
5502,how healthy are potato skins,1,nutrition_info
5503,how healthy is spaghetti,1,nutrition_info
5504,share the nutrition info for brownies with me,1,nutrition_info
...,...,...,...
14495,can you instruct me on how to make german choc...,104,recipe
14496,how do i make the perfect omelette,104,recipe
14497,i want to make sour dough bread please find a ...,104,recipe
14498,i need a really good recipe for making doughnuts,104,recipe


In [ ]:
def comprobar_balanceo(dict_datasets):
    """
    dict_datasets: Diccionario con formato {'Nombre': dataframe}
    Ejemplo: {'Train': cocina_train, 'Val': cocina_val, 'Test': cocina_test}
    """
    conteos = []

    for nombre, df in dict_datasets.items():
        # Obtenemos el conteo y lo convertimos a DataFrame
        c = df['intent_name'].value_counts().to_frame()
        c.columns = [nombre] # Renombramos la columna con el nombre del set
        conteos.append(c)

    # Unimos todos los conteos por el índice
    resumen = pd.concat(conteos, axis=1).fillna(0).astype(int)

    # Añadimos una columna de 'Total' para ver el peso global de cada clase
    resumen['Total'] = resumen.sum(axis=1)

    return resumen.sort_values(by='Total', ascending=False)

In [ ]:
# Metemos tus 3 datasets en un diccionario
mis_datasets = {
    'Train': cocina_train,
    'Val': cocina_val,
    'Test': cocina_test
}

# Ejecutamos la comprobación
df_balanceo = comprobar_balanceo(mis_datasets)
print(df_balanceo)

                         Train  Val  Test  Total
intent_name                                     
nutrition_info             100   20    30    150
food_last                  100   20    30    150
cook_time                  100   20    30    150
ingredient_substitution    100   20    30    150
calories                   100   20    30    150
ingredients_list           100   20    30    150
meal_suggestion            100   20    30    150
recipe                     100   20    30    150


Tenemos 8 clases perfectamente balanceadas.

## Combinando train + val. Apartamos test

ACTUALIZACIÓN: Ojeando test vemos que es mucho más diverso que train y val porque está literalmente pensado para ser el test. Así que bueno he decidido respetar eso y lo que hago es fusionar train + val (ligeramente menos datos, pero creo que por 10 datos menos por clase no nos morimos)

In [ ]:
mapeo = {
    'meal_suggestion': 1,
    'recipe': 2,
    'ingredients_list': 3,
    'ingredient_substitution': 4,
    'nutrition_info': 5,
    'calories': 6,
    'cook_time': 7,
    'food_last': 8
}

In [ ]:
dataset_train = pd.concat([cocina_train, cocina_val]).reset_index(drop=True)
dataset_train['intent'] = dataset_train['intent_name'].map(mapeo)
dataset_train

,text,intent,intent_name
0,what's the nutritional info for spaghetti,5,nutrition_info
1,what's the nutritional info for pizza,5,nutrition_info
2,how healthy are potato skins,5,nutrition_info
3,how healthy is spaghetti,5,nutrition_info
4,share the nutrition info for brownies with me,5,nutrition_info
...,...,...,...
955,do you have any good ways to make tomato soup,2,recipe
956,how do i make chicken alfredo,2,recipe
957,recipe for beef stroganoff,2,recipe
958,i need to find a good way to make chicken soup,2,recipe


In [ ]:
dataset_test = cocina_test.reset_index(drop=True)
dataset_test['intent'] = dataset_test['intent_name'].map(mapeo)
dataset_test[:20]

,text,intent,intent_name
0,tell me nutritional info for brocoli,5,nutrition_info
1,tell me nutritional info for lettuce,5,nutrition_info
2,tell me nutritional info for fish,5,nutrition_info
3,tell me nutritional info for burger,5,nutrition_info
4,tell me nutritional info for beans,5,nutrition_info
5,how healthy is blueberrys,5,nutrition_info
6,how healthy is tacos,5,nutrition_info
7,how healthy is mcdonalds,5,nutrition_info
8,how healthy is a cheeseburger,5,nutrition_info
9,how healthy is rice,5,nutrition_info


Veamos algunas muestras de cada clase.

In [ ]:
N = 20 # Número de ejemplos

for intencion in intenciones_cocina:
    print(f"==== Intención: {intencion} ====")

    # Filtramos el dataset para obtener solo las filas de la intención actual
    datos_filtrados = dataset_train[dataset_train['intent_name'] == intencion]

    # Tomamos los primeros N registros
    primeros = datos_filtrados.head(N)

    # Iteramos sobre esos registros para imprimir el texto
    for index, fila in primeros.iterrows():
        print(f"  {index}. {fila['text']}")
    print("")

==== Intención: meal_suggestion ====
  600. can you give me a french dinner suggestion
  601. suggest a meal from laos to me, please
  602. can you give me a vietnamese meal suggestion
  603. suggest a meal from burma to me
  604. can you give me a vietnamese dinner suggestion
  605. suggest a meal from thailand to me, please
  606. can you give me a thai meal suggestion, please
  607. can you give me a vietnamese meal suggestion, please
  608. can you give me a burmese meal suggestion, please
  609. suggest a meal from burma to me, please
  610. can you give me a burmese dinner suggestion
  611. can you give me a thai meal suggestion
  612. can you give me a thai dinner suggestion
  613. suggest a meal from thailand to me
  614. give me italian meal ideas
  615. suggest an italian meal for me
  616. can you suggest meals from italy to me
  617. may you suggest a meal from italy to me
  618. please suggest meals from italy to me
  619. can you tell me a good indian dish to make

==== I

## Traducir a español

In [ ]:
!pip install deep_translator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.4 MB/s eta 0:00:00


In [ ]:
from deep_translator import GoogleTranslator

# Configura el traductor
translator = GoogleTranslator(source='en', target='es')

def aumentar_idioma(df):
    # 0. Copia profunda para evitar warnings
    df = df.copy()
    df['language'] = 'en'

    # 1. Crear la copia para español
    df_es = df.copy()
    df_es['language'] = 'es'

    print(f"Traduciendo {len(df)} filas... (espera un momento)")

    # 2. Traducción usando apply estándar (sin progress_apply)
    # Convertimos a str(x) para evitar errores si hay celdas vacías
    df_es['text'] = df_es['text'].apply(lambda x: translator.translate(str(x)))

    # 3. Unir ambos datasets
    df_expandido = pd.concat([df, df_es], ignore_index=True)

    # 4. Re-indexar de 1 a N
    df_expandido.index = df_expandido.index + 1

    print("¡Listo!")
    return df_expandido

# Aplicar a tus datasets
train_final = aumentar_idioma(dataset_train)
test_final = aumentar_idioma(dataset_test)

Traduciendo 960 filas... (espera un momento)
¡Listo!
Traduciendo 240 filas... (espera un momento)
¡Listo!


Vale, esto ha tardado como 10 minutos. Vamos a guardar los datos de pandas en csv y así los podemos importar rápidamente el resto de veces.

In [ ]:
train_final.to_csv('data_train.csv', index=False)
test_final.to_csv('data_test.csv', index=False)

# PLN a partir de aquí (suponiendo datos cargados en Drive)

In [6]:
from google.colab import drive
drive.mount ('/content/drive')
%cd /content/drive/MyDrive/Colab Notebooks/PLN

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Colab Notebooks/PLN


In [7]:
import pandas as pd

# Cargar los conjuntos de datos
data_train = pd.read_csv('data_train.csv')
data_test = pd.read_csv('data_test.csv')

In [8]:
data_train

,text,intent,intent_name,language
0,what's the nutritional info for spaghetti,5,nutrition_info,en
1,what's the nutritional info for pizza,5,nutrition_info,en
2,how healthy are potato skins,5,nutrition_info,en
3,how healthy is spaghetti,5,nutrition_info,en
4,share the nutrition info for brownies with me,5,nutrition_info,en
...,...,...,...,...
1915,¿Tienes alguna buena manera de hacer sopa de t...,2,recipe,es
1916,como hago pollo alfredo,2,recipe,es
1917,receta de stroganoff de ternera,2,recipe,es
1918,Necesito encontrar una buena manera de hacer s...,2,recipe,es


Los dividimos según inglés y español.

In [9]:
data_train_es = data_train[data_train['language'] == 'es'].copy()
data_test_es = data_test[data_test['language'] == 'es'].copy()

data_train_en = data_train[data_train['language'] == 'en'].copy()
data_test_en = data_test[data_test['language'] == 'en'].copy()

Veamos traducciones inglés-español en test.

In [10]:
data_test_en

,text,intent,intent_name,language
0,tell me nutritional info for brocoli,5,nutrition_info,en
1,tell me nutritional info for lettuce,5,nutrition_info,en
2,tell me nutritional info for fish,5,nutrition_info,en
3,tell me nutritional info for burger,5,nutrition_info,en
4,tell me nutritional info for beans,5,nutrition_info,en
...,...,...,...,...
235,i need a pasta recipe,2,recipe,en
236,i want a recipe for roasted veggies,2,recipe,en
237,what is in a burrito recipe,2,recipe,en
238,give me a tuna salad recipe,2,recipe,en


In [11]:
data_test_es

,text,intent,intent_name,language
240,dime información nutricional del brócoli,5,nutrition_info,es
241,dime información nutricional de la lechuga,5,nutrition_info,es
242,dime información nutricional del pescado,5,nutrition_info,es
243,dime información nutricional de la hamburguesa,5,nutrition_info,es
244,dime información nutricional de los frijoles,5,nutrition_info,es
...,...,...,...,...
475,necesito una receta de pasta,2,recipe,es
476,quiero una receta de verduras asadas,2,recipe,es
477,¿Qué hay en una receta de burrito?,2,recipe,es
478,dame una receta de ensalada de atún,2,recipe,es


## Detección de idioma (no necesario en entrenamiento)

Para el entrenamiento no hay que detectar idioma, pero en la interfaz sí.

Opciones:
1. Un modelo que soporte ambos idiomas.
2. Dos modelos, uno para cada idioma. Ahora me están diciendo que es mejor lo 2do, así que empiezo por esto.

Esta librería nos haría el trabajo. Creo que sí utiliza n gramas de caracteres (quizás con esteroides). Esto habría que investigarlo obviamente mejor. [Documentación](https://pypi.org/project/lingua-language-detector/)

In [ ]:
# !pip install lingua-language-detector

In [ ]:
from lingua import Language, LanguageDetectorBuilder

detector = LanguageDetectorBuilder.from_languages(Language.SPANISH, Language.ENGLISH).build()

textos = ["komo hago pasta al dente", "how too bake poteito"]

for texto in textos:
    idioma = detector.detect_language_of(texto)
    print(f"Text: '{texto}' -> Language: {idioma.name}")

Text: 'komo hago pasta al dente' -> Language: SPANISH
Text: 'how too bake poteito' -> Language: ENGLISH


Lo bueno es que con faltas de ortografía funciona también.

## Preprocesamiento y normalización

Flujo de preprocesamiento y normalización

1. **Detección de idioma (Lingua):** Lingua analiza el input y determina si es español ("es") o inglés ("en"). Este resultado se pasa como el parámetro `lang`.
2. **Expansión de contracciones y Slang:** Limpia el texto en su forma "cruda" antes de que la IA de spaCy lo lea. Usa la librería contractions para expandir cosas como "I'm" a "I am". En español, usa tu diccionario SLANG_ES para reemplazar lenguaje de internet o abreviaturas (ej. "xq" se convierte en "porque", "kiero" en "quiero").
3. **Tokenización y limpieza básica (spaCy):** nlp(text) divide la frase en "tokens" (palabras individuales). Inmediatamente, tu código descarta los tokens que son espacios en blanco (is_space) o signos de puntuación (is_punct).
4. **Lematización (con excepciones):** Transforma cada palabra a su raíz o forma de diccionario (ej. "haciendo" -> "hacer", "manzanas" -> "manzana"). Antes de lematizar, quitas las tildes temporalmente para comprobar tu diccionario de EXCEPCIONES_LEMAS_ES. Si la palabra está ahí (como "gluten" o "vegano"), fuerzas a spaCy a usar tu lema personalizado para evitar que haga correcciones extrañas (como transformar "gluten" en el verbo "glutir").
5. **Corrección ortográfica sobre el lema (en producción solo):** Si estás en producción, usas pyspellchecker para revisar si el lema obtenido tiene faltas de ortografía. Como previamente cargaste una *whitelist* (`load_words`), el corrector ignora a propósito palabras clave de tu dominio que de otro modo marcaría como error (ej. "airfryer", "thermomix", "aove", "keto").
6. **Normalización final:** Fuera tildes y mayúsculas. Usando `unidecode.unidecode()`, eliminas definitivamente cualquier tilde o carácter especial que haya sobrevivido en el lema o que haya introducido el corrector ortográfico, y te aseguras de que todo esté en minúsculas.
7. **Filtrado inteligente de Stopwords:** Si la palabra está en tu lista de intocables (KEEP_ES / KEEP_EN), se guarda siempre (ej. "sin", "no", "vegano"). Esto es vital en recetas para no perder el significado de "sin gluten".
8. **Sinónimos/regionalismos:** Tras juntar todos los tokens limpios en una sola frase, la función apply_synonyms pasa un último filtro. Reemplaza ingredientes o términos regionales por un estándar unificado para que el modelo no tenga que aprender múltiples palabras para lo mismo (ej. transforma "papa" en "patata", "palta" en "aguacate", o "prawn" en "shrimp"). En inglés, también unifica bigramas (ej. "mange tout" -> "snow pea").


**Modelo predictivo**

El texto resultante (ej. "saludable ser espaguetis") es el que finalmente se envía a tu modelo de Machine Learning / IA para clasificar la intención.

In [1]:
!pip install spacy pyspellchecker lingua-language-detector

In [2]:
!pip install unidecode contractions

In [3]:
# Descargar el modelo de lenguaje en español
!python -m spacy download es_core_news_sm

# Descargar el modelo de lenguaje en inglés
!python -m spacy download en_core_web_sm

  Using cached https://github.com/explosion/spacy-models/releases/download/es_core_news_sm-3.8.0/es_core_news_sm-3.8.0-py3-none-any.whl (12.9 MB)
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [139]:
import re
import pandas as pd
import unidecode
import contractions
import spacy
from spellchecker import SpellChecker
from lingua import Language, LanguageDetectorBuilder

# ============================================================
# 1. INICIALIZACIÓN DE MODELOS
# ============================================================

print("Cargando modelos de spaCy y Lingua...")
nlp_es = spacy.load("es_core_news_sm")
nlp_en = spacy.load("en_core_web_sm")

# Detector acotado solo a ES y EN
detector = LanguageDetectorBuilder.from_languages(Language.SPANISH, Language.ENGLISH).build()

print("Cargando correctores ortográficos...")
spell_es = SpellChecker(language="es")
spell_en = SpellChecker(language="en")

# Whitelist: Palabras de cocina o extranjerismos que el corrector NO debe tocar
spell_es.word_frequency.load_words(["aove", "thermomix", "airfryer", "umami", "keto", "dente", "pizza",
                                    "sushi", "confit", "wok", "bowl", "tupper", "smoothie", "pancake", "crepe"])
spell_en.word_frequency.load_words(["bbq", "thermomix", "airfryer", "umami", "keto", "dente", "ramen"])

# ============================================================
# 2. DICCIONARIOS, STOPWORDS Y EXCEPCIONES
# ============================================================

KEEP_ES = {
    "no", "sin", "excepto", "nunca", "jamas",
    "cuando", "como", "cuanto", "antes", "despues",
    "hacer",
    "estar", "ser", "buen", "mal",
    "gluten", "lactosa", "proteina", "vegano", "vegana", "keto", "umami"
}
KEEP_EN = {
    "no", "not", "without", "never", "free",
    "when", "how", "before", "after",
    "make",
    "good", "bad",
    "gluten", "lactose", "vegan", "keto", "umami", "bbq"
}

# Stopwords limpias sin tildes
STOPWORDS_ES = {unidecode.unidecode(w).lower() for w in nlp_es.Defaults.stop_words} - KEEP_ES
STOPWORDS_EN = {unidecode.unidecode(w).lower() for w in nlp_en.Defaults.stop_words} - KEEP_EN

# Slang de internet (se limpia antes de hacer nada)
SLANG_ES = {
    "aser": "hacer", "kiero": "quiero", "q": "que", "xq": "porque",
    "komo": "como", "weno": "bueno", "k": "que", "x": "por"
}

# Excepciones spaCy
EXCEPCIONES_LEMAS_ES = {
    "gluten": "gluten", "proteina": "proteina", "lactosa": "lactosa",
    "vegano": "vegano", "vegana": "vegano", "bizcocho": "bizcocho",
    "aove": "aove", "pizza": "pizza"
}

EXCEPCIONES_LEMAS_EN = {
    "gluten": "gluten", "ramen": "ramen",
    "vegan": "vegan"
}

# Sinónimos
SYNONYMS_ES = {
    "papa": "patata", "papas": "patata", "pure de papas": "pure patata",
    "durazno": "melocoton", "duraznos": "melocoton",
    "chicharo": "guisante", "chicharos": "guisante",
    "palta": "aguacate", "paltas": "aguacate", "palto": "aguacate",
    "frutilla": "fresa", "frutillas": "fresa",
    "poroto": "judia", "porotos": "judia",
    "choclo": "maiz", "choclos": "maiz", "elote": "maiz",
    "anana": "pina",
    "jugo": "zumo",
    "banana": "platano", "bananas": "platano",
    "morron": "pimiento", "morrones": "pimiento",
    "zapallo": "calabaza", "zapallos": "calabaza",
    "aji": "chile", "ajies": "chile",
    "mani": "cacahuete", "manies": "cacahuete",
    "camaron": "gamba", "camarones": "gamba",
    "jitomate": "tomate", "jitomates": "tomate",
    "batata": "boniato", "batatas": "boniato"
}

SYNONYMS_EN = {
    "aubergine": "eggplant", "aubergines": "eggplant",
    "courgette": "zucchini", "courgettes": "zucchini",
    "coriander": "cilantro",
    "capsicum": "bell pepper",
    "rocket": "arugula",
    "biscuit": "cookie", "biscuits": "cookie",
    "yoghurt": "yogurt",
    "beetroot": "beet", "beetroots": "beet",
    "mangetout": "snow pea",
    "swede": "turnip",
    "prawn": "shrimp", "prawns": "shrimp",
}
BIGRAMS_EN = {"mange tout": "snow pea"}

# ============================================================
# 3. MOTOR DE PROCESAMIENTO
# ============================================================

def apply_synonyms(text: str, lang: str) -> str:
    """Aplica sinónimos al texto ya limpio."""
    if lang == "en":
        for bigram, canon in BIGRAMS_EN.items():
            text = text.replace(bigram, canon)
    synonyms = SYNONYMS_ES if lang == "es" else SYNONYMS_EN
    return " ".join(synonyms.get(w, w) for w in text.split())

def procesar_texto(text: str, lang: str, is_predict=False) -> str:
    """
    is_predict=False -> Úsalo en tu CSV de entrenamiento (rápido, sin corrector).
    is_predict=True  -> Úsalo en la app final (con corrector ortográfico).
    """
    if not isinstance(text, str) or not text.strip():
        return ""

    # 1. Expandir contracciones y Slang
    if lang == "en":
        text = contractions.fix(text)
    if lang == "es":
        for slang, correccion in SLANG_ES.items():
            text = re.sub(rf'\b{slang}\b', correccion, text, flags=re.IGNORECASE)

    # 2. Configurar herramientas según idioma
    nlp = nlp_es if lang == "es" else nlp_en
    stopwords = STOPWORDS_ES if lang == "es" else STOPWORDS_EN
    keep_set = KEEP_ES if lang == "es" else KEEP_EN
    spell = spell_es if lang == "es" else spell_en

    # 3. Lematización inicial con spaCy
    doc = nlp(text)
    tokens_limpios = []

    for token in doc:
        if token.is_space or token.is_punct:
            continue

        texto_sin_tildes = unidecode.unidecode(token.text).lower()

        # 4. Obtener lema protegiendo excepciones ("glutir")
        if lang == "es" and texto_sin_tildes in EXCEPCIONES_LEMAS_ES:
            lemma_con_tildes = EXCEPCIONES_LEMAS_ES[texto_sin_tildes]
        elif lang == "en" and texto_sin_tildes in EXCEPCIONES_LEMAS_EN:
            lemma_con_tildes = EXCEPCIONES_LEMAS_EN[texto_sin_tildes]
        else:
            lemma_con_tildes = token.lemma_.lower()

        # 5. Corrector Ortográfico SOBRE EL LEMA (Solo en Producción)
        if is_predict:
            if spell.unknown([lemma_con_tildes]):
                correccion = spell.correction(lemma_con_tildes)
                lemma_con_tildes = correccion if correccion else lemma_con_tildes

        # 6. Normalización final (Fuera tildes)
        lemma_norm = unidecode.unidecode(lemma_con_tildes)

        # 7. Filtrado de Stopwords
        if lemma_norm in keep_set:
            tokens_limpios.append(lemma_norm)
        elif lemma_norm not in stopwords and len(lemma_norm) > 1:
            tokens_limpios.append(lemma_norm)

    texto_procesado = " ".join(tokens_limpios)

    # 8. Aplicar sinónimos (ej. "papa" -> "patata")
    return apply_synonyms(texto_procesado, lang)

Cargando modelos de spaCy y Lingua...
Cargando correctores ortográficos...


In [140]:
# --- ESPAÑOL ---
print("Limpiando textos en español...")
data_train_es['clean_text'] = data_train_es['text'].apply(lambda x: procesar_texto(str(x), lang='es', is_predict=False))
data_test_es['clean_text'] = data_test_es['text'].apply(lambda x: procesar_texto(str(x), lang='es', is_predict=False))

# --- INGLÉS ---
print("Limpiando textos en inglés...")
data_train_en['clean_text'] = data_train_en['text'].apply(lambda x: procesar_texto(str(x), lang='en', is_predict=False))
data_test_en['clean_text'] = data_test_en['text'].apply(lambda x: procesar_texto(str(x), lang='en', is_predict=False))

print("¡Todo listo!")

Limpiando textos en español...
Limpiando textos en inglés...
¡Todo listo!


In [141]:
data_train_en

,text,intent,intent_name,language,clean_text
0,what's the nutritional info for spaghetti,5,nutrition_info,en,nutritional info spaghetti
1,what's the nutritional info for pizza,5,nutrition_info,en,nutritional info pizza
2,how healthy are potato skins,5,nutrition_info,en,how healthy potato skin
3,how healthy is spaghetti,5,nutrition_info,en,how healthy spaghetti
4,share the nutrition info for brownies with me,5,nutrition_info,en,share nutrition info brownie
...,...,...,...,...,...
955,do you have any good ways to make tomato soup,2,recipe,en,good way make tomato soup
956,how do i make chicken alfredo,2,recipe,en,how make chicken alfredo
957,recipe for beef stroganoff,2,recipe,en,recipe beef stroganoff
958,i need to find a good way to make chicken soup,2,recipe,en,need find good way make chicken soup


In [142]:
data_train_es

,text,intent,intent_name,language,clean_text
960,¿Cuál es la información nutricional de los esp...,5,nutrition_info,es,ser informacion nutricional espaguetis
961,¿Cuál es la información nutricional de la pizza?,5,nutrition_info,es,ser informacion nutricional pizza
962,¿Qué tan saludables son las pieles de papa?,5,nutrition_info,es,saludable ser piel patata
963,¿Qué tan saludables son los espaguetis?,5,nutrition_info,es,saludable ser espaguetis
964,comparte conmigo la información nutricional de...,5,nutrition_info,es,compartir informacion nutricional browni
...,...,...,...,...,...
1915,¿Tienes alguna buena manera de hacer sopa de t...,2,recipe,es,buen hacer sopa tomate
1916,como hago pollo alfredo,2,recipe,es,como hacer pollo alfredo
1917,receta de stroganoff de ternera,2,recipe,es,receta stroganoff ternera
1918,Necesito encontrar una buena manera de hacer s...,2,recipe,es,necesitar encontrar buen hacer sopa pollo


In [143]:
data_train_es.to_csv('clean_data_train_es.csv', index=False)
data_test_es.to_csv('clean_data_test_es.csv', index=False)

data_train_en.to_csv('clean_data_train_en.csv', index=False)
data_test_en.to_csv('clean_data_test_en.csv', index=False)

## Interfaz (de prueba)

Podemos probar más cosas y ver qué tal.

In [144]:
# ============================================================
# EL ENRUTADOR DE PRODUCCIÓN
# ============================================================

detector = LanguageDetectorBuilder.from_languages(Language.SPANISH, Language.ENGLISH).build()

def procesar_peticion_usuario(texto_usuario: str):
    """Función para el usuario final (Interfaz Gráfica / API)"""
    idioma_lingua = detector.detect_language_of(texto_usuario)
    lang_code = "es" if idioma_lingua == Language.SPANISH else "en"

    # Activa is_predict=True para usar el corrector ortográfico
    texto_limpio = procesar_texto(texto_usuario, lang=lang_code, is_predict=True)

    return lang_code, texto_limpio

# ============================================================
# PRUEBAS FINALES
# ============================================================

print("Test de producción (pipeline en acción)\n")

test_cases = [
    "kiero comer super rico algo japones",
    "i wana make fish and chips",
    "torta de aji y jitomate",
    "can i sustitute eggplant with Fresh BASIL",
    "mañana me suven el seldo y ma dicho mi amiga que después aciendo tarta de chocolate",
    "quiero verduras al wok pa meter en un tupper o un bowl y luego hacer batata y boniato"
]

for texto in test_cases:
    idioma, limpio = procesar_peticion_usuario(texto)
    print(f"[{idioma.upper()}] In : {texto}")
    print(f"[{idioma.upper()}] Out: {limpio}\n")

Test de producción (pipeline en acción)

[ES] In : kiero comer super rico algo japones
[ES] Out: querer comer super rico japon

[EN] In : i wana make fish and chips
[EN] Out: want make fish chip

[ES] In : torta de aji y jitomate
[ES] Out: torta chile tomate

[EN] In : can i sustitute eggplant with Fresh BASIL
[EN] Out: substitute eggplant fresh basil

[ES] In : mañana me suven el seldo y ma dicho mi amiga que después aciendo tarta de chocolate
[ES] Out: manana subir sueldo amiga despues hacer tarta chocolate

[ES] In : quiero verduras al wok pa meter en un tupper o un bowl y luego hacer batata y boniato
[ES] Out: querer verdura wok meter tupper bowl hacer boniato boniato



## Clasificación de intención

Esto es tf-idf + LinearSVC con parámetros sin optimizar. Se podría probar a ajusta el modelo a ver si mejora y otros algoritmos de clasificación (Naive Bayes multinomial, Logística, Árboles, etc).

In [145]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report

# 1. Crear el vectorizador y el modelo para ESPAÑOL
vectorizer_es = TfidfVectorizer()
model_svc_es = LinearSVC(C=1, max_iter=1000, random_state=42)

# 2. Vectorizar los datos de ENTRENAMIENTO (usamos fit_transform)
X_train_es = vectorizer_es.fit_transform(data_train_es['clean_text'])
y_train_es = data_train_es['intent']

# 3. Entrenar el modelo con el 100% del dataset de train
print("Entrenando modelo en español...")
model_svc_es.fit(X_train_es, y_train_es)

# 4. Vectorizar los datos de TEST (¡ATENCIÓN! Aquí usamos solo .transform)
# Es vital que el test set se limpie exactamente igual que el train set
X_test_es = vectorizer_es.transform(data_test_es['clean_text'])
y_test_es = data_test_es['intent']

# 5. Evaluar el modelo
predicciones_es = model_svc_es.predict(X_test_es)
print("\n=== Resultados del modelo en español sobre test ===")
print(classification_report(y_test_es, predicciones_es))

Entrenando modelo en español...

=== Resultados del modelo en español sobre test ===
              precision    recall  f1-score   support

           1       0.96      0.90      0.93        30
           2       0.81      0.97      0.88        30
           3       0.96      0.83      0.89        30
           4       0.97      0.97      0.97        30
           5       0.97      1.00      0.98        30
           6       1.00      0.97      0.98        30
           7       0.97      0.93      0.95        30
           8       0.97      1.00      0.98        30

    accuracy                           0.95       240
   macro avg       0.95      0.95      0.95       240
weighted avg       0.95      0.95      0.95       240



In [146]:
# 1. Crear el vectorizador y el modelo para INGLÉS
vectorizer_en = TfidfVectorizer()
model_svc_en = LinearSVC(C=1, max_iter=1000, random_state=42)

# 2. Vectorizar los datos de ENTRENAMIENTO (usamos fit_transform)
X_train_en = vectorizer_en.fit_transform(data_train_en['clean_text'])
y_train_en = data_train_en['intent']

# 3. Entrenar el modelo con el 100% del dataset de train
print("Entrenando modelo en inglés...")
model_svc_en.fit(X_train_en, y_train_en)

# 4. Vectorizar los datos de TEST (¡ATENCIÓN! Aquí usamos solo .transform)
# Es vital que el test set se limpie exactamente igual que el train set
X_test_en = vectorizer_en.transform(data_test_en['clean_text'])
y_test_en = data_test_en['intent']

# 5. Evaluar el modelo
predicciones_en = model_svc_en.predict(X_test_en)
print("\n=== Resultados del modelo en inglés sobre test ===")
print(classification_report(y_test_en, predicciones_en))

Entrenando modelo en inglés...

=== Resultados del modelo en inglés sobre test ===
              precision    recall  f1-score   support

           1       0.93      0.90      0.92        30
           2       0.80      0.80      0.80        30
           3       0.90      0.87      0.88        30
           4       1.00      0.97      0.98        30
           5       0.97      1.00      0.98        30
           6       1.00      0.97      0.98        30
           7       0.85      0.97      0.91        30
           8       1.00      0.97      0.98        30

    accuracy                           0.93       240
   macro avg       0.93      0.93      0.93       240
weighted avg       0.93      0.93      0.93       240



## Prueba final

In [147]:
# ============================================================
# 1. MAPEO DE INTENCIONES (INTENT MAPPING)
# ============================================================

intent_mapping = {
    'meal_suggestion': 1,
    'recipe': 2,
    'ingredients_list': 3,
    'ingredient_substitution': 4,
    'nutrition_info': 5,
    'calories': 6,
    'cook_time': 7,
    'food_last': 8
}

# Le damos la vuelta para buscar por número: {1: 'meal_suggestion', 2: 'recipe', ...}
reverse_intent_mapping = {num: name for name, num in intent_mapping.items()}

# ============================================================
# 2. FUNCIÓN DE PREDICCIÓN (PREDICTION FUNCTION)
# ============================================================

def predict_intent(user_message):
    """
    Toma un mensaje crudo del usuario, detecta el idioma, lo limpia,
    lo vectoriza y predice la intención usando el modelo adecuado.
    """
    # 1. Seguridad: Verificar que el mensaje no esté vacío
    if not isinstance(user_message, str) or not user_message.strip():
        return {"error": "Empty message", "intent": None, "lang": None}

    # 2. Detectar el idioma con Lingua
    detected_language = detector.detect_language_of(user_message)

    # 3. Enrutar al modelo correcto según el idioma
    # (Nota: Asegúrate de que tus variables globales ahora se llamen así)
    if detected_language == Language.ENGLISH:
        lang_code = "en"
        vectorizer = vectorizer_en
        model = model_svc_en
    else:
        # Por defecto (o si detecta Language.SPANISH) usamos el motor en español
        lang_code = "es"
        vectorizer = vectorizer_es
        model = model_svc_es

    # 4. Limpiar y procesar el texto (¡Activamos el corrector ortográfico con is_predict=True!)
    clean_text = procesar_texto(user_message, lang=lang_code, is_predict=True)

    # Si el mensaje tras limpiarlo se queda vacío (ej. solo mandó emojis o stopwords)
    if not clean_text:
         return {"error": "Unrecognizable text after cleaning", "intent": None, "lang": lang_code}

    # 5. Vectorizar el texto limpio (¡IMPORTANTE: Solo usamos .transform!)
    text_vector = vectorizer.transform([clean_text])

    # 6. Predecir la intención (devuelve el número como entero)
    # .predict() devuelve una matriz/lista, tomamos el elemento [0]
    predicted_intent_num = int(model.predict(text_vector)[0])

    # 7. Buscar el nombre de la intención en nuestro mapeo inverso
    intent_name = reverse_intent_mapping.get(predicted_intent_num, "unknown")

    # Formatear la intención final (ej. "2 (recipe)")
    formatted_intent = f"{predicted_intent_num} ({intent_name})"

    # 8. Retornar los resultados en un formato limpio (ideal para una API)
    return {
        "original_text": user_message,
        "clean_text": clean_text,
        "detected_language": lang_code,
        "predicted_intent": formatted_intent
    }

Vamos a probar. Recordemos las intenciones.

In [157]:
data_train_es['intent_name'].unique()

array(['nutrition_info', 'food_last', 'cook_time',
       'ingredient_substitution', 'calories', 'ingredients_list',
       'meal_suggestion', 'recipe'], dtype=object)

Haciendo pruebas muy dirigidas va decente. Pero si eres demasiado escueto le cuesta y confunde categorías. Recordemos que tenemos solo 1000 datos aprox por idioma y que además estamos usando un modelo básico sin ajustar hiperparámetros. Habría que explorar modelos básicos y ajustar hiperparámetros.

Por otro lado, como alternativa, podríamos probar con un transformer no demasiado pesado. El profesor utiliza en el ejemplo dos vías por explorar, una simple y un transformer y se queda tan a gusto.

Tambiñen hay que ver cómo contextualizar palabras porque por ahora tenemos intenciones, y cómo llevar al FAQ. Que si whoosh y búsqueda indexada, o no sé como. Pero pusimos en la entrega 1 cosas de filtro de intención y luego con embeddings similitud coseno y algo así no sé.

### Ejemplos guiados

In [148]:
# Variar mensaje para ir probando
predict_intent("cuando se me pone malo el queso fresco?")

{'original_text': 'cuando se me pone malo el queso fresco?',
 'clean_text': 'cuando malo queso fresco',
 'detected_language': 'es',
 'predicted_intent': '8 (food_last)'}

In [149]:
predict_intent("si quiero no tener que cocinar mucho, que recetas ligeras me recomiendas")

{'original_text': 'si quiero no tener que cocinar mucho, que recetas ligeras me recomiendas',
 'clean_text': 'querer no cocinar receta ligero recomendar',
 'detected_language': 'es',
 'predicted_intent': '2 (recipe)'}

In [150]:
predict_intent("necesito no usar leche de vaca para mejor usar leche de coco")

{'original_text': 'necesito no usar leche de vaca para mejor usar leche de coco',
 'clean_text': 'necesitar no leche vaca leche coco',
 'detected_language': 'es',
 'predicted_intent': '4 (ingredient_substitution)'}

In [151]:
predict_intent("la verdad que me gustaria hacer comida italiana para cenar que me recomiendas")

{'original_text': 'la verdad que me gustaria hacer comida italiana para cenar que me recomiendas',
 'clean_text': 'gustariar hacer comida italiano cenar recomendar',
 'detected_language': 'es',
 'predicted_intent': '1 (meal_suggestion)'}

In [154]:
predict_intent("cuanto tiempo tarda en hacerse el pollo al horno")

{'original_text': 'cuanto tiempo tarda en hacerse el pollo al horno',
 'clean_text': 'cuanto tiempo tardar hacer el pollo horno',
 'detected_language': 'es',
 'predicted_intent': '7 (cook_time)'}

In [162]:
predict_intent("que ingredientes necesito para hacer pollo la horno")

{'original_text': 'que ingredientes necesito para hacer pollo la horno',
 'clean_text': 'ingrediente necesitar hacer pollo horno',
 'detected_language': 'es',
 'predicted_intent': '3 (ingredients_list)'}

In [164]:
predict_intent("puedo comer queso fundido todos los dias sin ponerme mala")

{'original_text': 'puedo comer queso fundido todos los dias sin ponerme mala',
 'clean_text': 'comer queso fundido sin poner yo malo',
 'detected_language': 'es',
 'predicted_intent': '5 (nutrition_info)'}

In [167]:
predict_intent("cuantas calorias tiene el pollo frito")

{'original_text': 'cuantas calorias tiene el pollo frito',
 'clean_text': 'caloria pollo frito',
 'detected_language': 'es',
 'predicted_intent': '6 (calories)'}

### Ejemplos difíciles

In [168]:
predict_intent("el pollo frito engorda mucho?")

{'original_text': 'el pollo frito engorda mucho?',
 'clean_text': 'pollo frito engordar',
 'detected_language': 'es',
 'predicted_intent': '2 (recipe)'}

Cómo que receta xd

In [169]:
predict_intent("que lleva una fajita mexicana")

{'original_text': 'que lleva una fajita mexicana',
 'clean_text': 'frito mejicano',
 'detected_language': 'es',
 'predicted_intent': '4 (ingredient_substitution)'}

no comments

In [174]:
predict_intent("es mejor comer brocoli o comer patatas fritas para estar sano")

{'original_text': 'es mejor comer brocoli o comer patatas fritas para estar sano',
 'clean_text': 'ser comer brecol comer patata frito estar sano',
 'detected_language': 'es',
 'predicted_intent': '8 (food_last)'}